# Multistage Training Pipeline — three teams, Registry-linked

Three independent training "teams" collaborate on one model, with
the **W&B Registry** as the sole communication surface. No shared
Python state, no filesystem, no messaging channels — every team
reads a Registry Collection alias and writes to a different one:

1. **Stage 1 — init team.** Consumes
   `hilift_multistage_models/base:production` (untrained Transolver++)
   + `hilift_multistage_datasets/synthetic:production`. Derives the
   **canonical normalisation stats** on the synthetic split and
   bakes them into the output checkpoint. Promotes to
   `hilift_multistage_models/initialised:production`.
2. **Stage 2 — pretraining team.** Consumes `initialised:production`
   + `datasets/pretraining:production`. **Reuses** Stage 1's stats
   from the checkpoint — no recomputation. Trains on the broader
   3-geometry × low/mid-AoA domain. Promotes to
   `pretrained:production`.
3. **Stage 3 — posttraining team.** Consumes `pretrained:production`
   + `datasets/posttraining:production`. Same stats propagation.
   Fine-tunes on the high-AoA / near-stall regime — the target
   deployment domain. Promotes to `posttrained:production` (final).

Each stage's `wandb.init(job_type="train-stage")` block owns its
own run — no `resume=`, no touching upstream runs. Lineage is
carried entirely by `use_artifact` (input edges) + `link_artifact`
(output edges) on the four Registry Collections.

![](../docs/explainers/multistage_diagram.png)

**Prerequisites**
- `advanced_multistage_setup.ipynb` executed first — publishes the
  `base` model and the three datasets.

## 1. Setup

Third-party imports and section-local constants.


In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import wandb

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat()


In [ ]:
# ---- W&B target ----
ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

# ---- Registries + Collections (must match advanced_multistage_setup.ipynb) ----
DATASETS_REG = "hilift_multistage_datasets"
MODELS_REG   = "hilift_multistage_models"

# ---- Training shape (matches experiment_tracking_hilift.ipynb) ----
PATCH_POINTS = 1024
PATCHES_PER_CASE = 96
BATCH_SIZE = 4

# ---- Val split ----
VAL_FRAC = 0.2

TMP_DIR = Path(tempfile.mkdtemp(prefix="hilift_multistage_pipe_"))

print(f"torch    = {torch.__version__}")
print(f"wandb    = {wandb.__version__}")
print(f"device   = {device}")
print(f"tmp      = {TMP_DIR}")


## 2. Shared model + `train_one_stage` helper

One cell holds everything all three stages reuse: the Transolver++
clone/patch (with cached-repo re-use), the
`TransolverPointFieldSurrogate` wrapper, the patch-based
`HiLiftPointPatchDataset`, the `_load_npz_case` reader, the
`aoa_condition` encoder, plus the reusable `train_one_stage`
function that each stage below calls once.

`train_one_stage` handles the full stage lifecycle:

1. Opens a `wandb.init(job_type="train-stage")` run.
2. `use_artifact`s both the source model and the dataset — creates
   the input lineage edges.
3. Loads the source checkpoint's `state_dict`, `arch`, and
   (Stages 2/3) `stats`.
4. **`derive_stats=True`** (Stage 1 only) computes fresh
   normalisation stats from the training split; otherwise the
   source checkpoint's stats are reused unchanged.
5. Builds patch-based train/val DataLoaders.
6. Trains for `epochs` epochs with a cosine schedule.
7. Saves the output checkpoint (state_dict + arch + stats + stage
   metadata + epoch history).
8. Logs + `link_artifact`s the output to the target Collection with
   aliases `production`, `v1`.
9. **Writes a Collection card** — the `collection_description`
   markdown is set on the target Collection via the Public API
   (`api.artifact_collection(...).description = "..."; .save()`).
   Idempotent: re-running a stage re-syncs the card, so the cards
   below stay in lock-step with whatever ships in code.

Returns the linked Registry path so the caller can chain stages.


In [ ]:
# ==== §2 — Transolver++ + Dataset + helpers + train_one_stage ==========

# ---- Transolver++ clone + patch ----
CACHE_ROOT = Path.home() / ".cache" / "physicsx_demo"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
TRANSOLVER_DIR = CACHE_ROOT / "Transolver_plus"

if not TRANSOLVER_DIR.exists():
    print(f"Cloning thuml/Transolver_plus into {TRANSOLVER_DIR} ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/thuml/Transolver_plus.git",
        str(TRANSOLVER_DIR),
    ])
else:
    print(f"Using cached Transolver_plus at {TRANSOLVER_DIR}")

model_file = TRANSOLVER_DIR / "models" / "Transolver_plus.py"
src_text = model_file.read_text()
if "def _maybe_all_reduce" not in src_text:
    src_text = src_text.replace(
        "import torch.distributed.nn as dist_nn\n",
        "import torch.distributed.nn as dist_nn\n"
        "import torch.distributed as dist\n\n\n"
        "def _maybe_all_reduce(tensor):\n"
        "    if dist.is_available() and dist.is_initialized() "
        "and dist.get_world_size() > 1:\n"
        "        dist_nn.all_reduce(tensor, op=dist_nn.ReduceOp.SUM)\n"
        "    return tensor\n",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_norm, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_norm)",
    )
    src_text = src_text.replace(
        "dist_nn.all_reduce(slice_token, op=dist_nn.ReduceOp.SUM)",
        "_maybe_all_reduce(slice_token)",
    )
    model_file.write_text(src_text)
    print("Patched Transolver_plus.py with _maybe_all_reduce.")
else:
    print("Transolver_plus.py already patched.")

if str(TRANSOLVER_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSOLVER_DIR))
if "models.Transolver_plus" in sys.modules:
    importlib.reload(sys.modules["models.Transolver_plus"])
from models.Transolver_plus import Model as TransolverPlusModel


# ---- Model wrapper ----
class TransolverPointFieldSurrogate(nn.Module):
    """Point-cloud → per-point scalar prediction via Transolver++."""

    def __init__(
        self,
        feature_dim: int = 6,
        out_dim: int = 1,
        n_hidden: int = 96,
        n_layers: int = 4,
        n_head: int = 4,
        slice_num: int = 16,
        mlp_ratio: int = 2,
        dropout: float = 0.05,
    ) -> None:
        super().__init__()
        self.backbone = TransolverPlusModel(
            space_dim=feature_dim, fun_dim=0, out_dim=out_dim,
            n_hidden=n_hidden, n_layers=n_layers, n_head=n_head,
            mlp_ratio=mlp_ratio, slice_num=slice_num,
            dropout=dropout, unified_pos=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        condition: torch.Tensor,
    ) -> torch.Tensor:
        return self.backbone((x, pos, condition))


# ---- Data helpers ----
def _load_npz_case(path: Path) -> dict:
    """Load one setup-notebook NPZ back into a plain-Python dict."""
    data = np.load(path, allow_pickle=True)
    case = {k: data[k] for k in data.files}
    for key in ["geo_id", "run_id", "target_name",
                "target_location", "target_mode"]:
        if key in case:
            case[key] = (
                str(case[key].item()) if case[key].ndim == 0 else str(case[key])
            )
    case["aoa"] = float(np.asarray(case["aoa"]).item())
    return case


def load_dataset_cases(registry_path: str) -> list[dict]:
    """Download a dataset artifact from the Registry and load its NPZs."""
    api = wandb.Api()
    art = api.artifact(registry_path)
    d = Path(art.download())
    return [_load_npz_case(p) for p in sorted(d.glob("*.npz"))]


def split_train_val(
    cases: list[dict], val_frac: float,
) -> tuple[list[dict], list[dict]]:
    """Case-level 80/20 split (deterministic; at least 1 val case)."""
    rng = np.random.default_rng(SEED)
    order = np.arange(len(cases))
    rng.shuffle(order)
    n_val = max(1, int(round(val_frac * len(cases))))
    n_val = min(n_val, len(cases) - 1)     # always keep at least 1 train case
    val = [cases[i] for i in order[:n_val]]
    train = [cases[i] for i in order[n_val:]]
    return train, val


def derive_stats(train_cases: list[dict]) -> dict:
    """Compute point/target/AoA normalisation stats on the training split."""
    train_points = np.concatenate([c["points"] for c in train_cases], axis=0)
    train_target = np.concatenate([c["target"] for c in train_cases], axis=0)
    return {
        "point_mean":  train_points.mean(axis=0).astype(np.float32),
        "point_std":   np.maximum(train_points.std(axis=0), 1e-6).astype(np.float32),
        "target_mean": train_target.mean(axis=0).astype(np.float32),
        "target_std":  np.maximum(train_target.std(axis=0), 1e-6).astype(np.float32),
        "aoa_mean":    np.float32(np.mean([c["aoa"] for c in train_cases])),
        "aoa_std":     np.float32(max(np.std([c["aoa"] for c in train_cases]), 1.0)),
    }


def aoa_condition(aoa_degrees: float, stats: dict) -> np.ndarray:
    """Encode angle of attack as (z-norm, sin(rad), cos(rad))."""
    aoa_norm = (np.float32(aoa_degrees) - stats["aoa_mean"]) / stats["aoa_std"]
    aoa_rad = np.deg2rad(np.float32(aoa_degrees))
    return np.array(
        [aoa_norm, np.sin(aoa_rad), np.cos(aoa_rad)],
        dtype=np.float32,
    )


class HiLiftPointPatchDataset(Dataset):
    """Yield random `patch_points`-sized subsets of each case per epoch."""

    def __init__(
        self,
        case_list: list[dict],
        stats: dict,
        patch_points: int,
        patches_per_case: int,
        random_patches: bool = True,
    ) -> None:
        self.case_list = case_list
        self.stats = stats
        self.patch_points = patch_points
        self.patches_per_case = patches_per_case
        self.random_patches = random_patches
        self.rng = np.random.default_rng(SEED)

    def __len__(self) -> int:
        return len(self.case_list) * self.patches_per_case

    def __getitem__(self, idx: int):
        case = self.case_list[idx % len(self.case_list)]
        n = len(case["points"])
        replace = n < self.patch_points

        if self.random_patches:
            point_idx = self.rng.choice(n, size=self.patch_points, replace=replace)
        else:
            rng = np.random.default_rng(SEED + idx)
            point_idx = rng.choice(n, size=self.patch_points, replace=replace)

        points = case["points"][point_idx].astype(np.float32)
        normals = case["normals"][point_idx].astype(np.float32)
        target = case["target"][point_idx].astype(np.float32)

        pos = (points - self.stats["point_mean"]) / self.stats["point_std"]
        y = (target - self.stats["target_mean"]) / self.stats["target_std"]

        x = np.concatenate([pos, normals], axis=1).astype(np.float32)
        cond = aoa_condition(case["aoa"], self.stats)

        return (
            torch.tensor(x),
            torch.tensor(pos.astype(np.float32)),
            torch.tensor(cond),
            torch.tensor(y.astype(np.float32)),
        )


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    stats: dict,
) -> tuple[float, float, float]:
    """Return (loss, mae, rmse) on the val loader."""
    model.eval()
    torch.manual_seed(SEED)
    target_mean_t = torch.tensor(stats["target_mean"], dtype=torch.float32, device=device)
    target_std_t  = torch.tensor(stats["target_std"],  dtype=torch.float32, device=device)

    losses, maes, rmses = [], [], []
    for x, pos, cond, y in loader:
        x, pos, cond, y = x.to(device), pos.to(device), cond.to(device), y.to(device)
        pred = model(x, pos, cond)
        loss = F.mse_loss(pred, y)
        pred_phys = pred * target_std_t + target_mean_t
        y_phys    = y    * target_std_t + target_mean_t
        err = pred_phys - y_phys
        losses.append(loss.item())
        maes.append(err.abs().mean().item())
        rmses.append(torch.sqrt((err ** 2).mean()).item())
    return float(np.mean(losses)), float(np.mean(maes)), float(np.mean(rmses))


# ==== The core stage function ==========================================
def train_one_stage(
    source_model_ref: str,
    dataset_ref: str,
    stage: str,
    target_collection: str,
    epochs: int,
    lr: float,
    derive_stats_from_data: bool,
    collection_description: str,
) -> str:
    """Run one pipeline stage; return the linked Registry path.

    Parameters
    ----------
    source_model_ref : str
        Registry path of the input model, e.g.
        `wandb-registry-hilift_multistage_models/base:production`.
    dataset_ref : str
        Registry path of the input dataset.
    stage : str
        Label for the output ("initialised" | "pretrained" | "posttrained").
    target_collection : str
        Collection name under `hilift_multistage_models` to link into.
    epochs, lr : int, float
        Stage-specific training budget.
    derive_stats_from_data : bool
        Stage 1 only: derive fresh normalisation stats. Stages 2/3
        must pass False so upstream stats propagate unchanged.
    collection_description : str
        Markdown card written to the target Collection after linking.
        Rendered on the Registry Collection page; idempotent, so
        re-running a stage re-syncs the card with the code.
    """
    with wandb.init(
        entity=ENTITY, project=PROJECT,
        job_type="train-stage",
        name=f"hilift-multistage-{stage}",
        group="hilift-multistage",
        resume="never",
        config={
            "stage":             stage,
            "source_model_ref":  source_model_ref,
            "dataset_ref":       dataset_ref,
            "target_collection": target_collection,
            "epochs":            epochs,
            "lr":                lr,
            "patch_points":      PATCH_POINTS,
            "patches_per_case":  PATCHES_PER_CASE,
            "batch_size":        BATCH_SIZE,
            "derive_stats":      derive_stats_from_data,
            "started_at":        utc_ts(),
        },
        tags=["notebook", "advanced", "multistage", stage],
    ) as run:

        # ---- Input lineage edges ----
        src_model_art = run.use_artifact(source_model_ref, type="model")
        src_ds_art    = run.use_artifact(dataset_ref, type="dataset")

        # ---- Load source checkpoint (state_dict + arch [+ stats]) ----
        d = Path(src_model_art.download())
        src_ckpt = torch.load(
            next(d.glob("*.pt")),
            map_location=device, weights_only=False,
        )
        arch = dict(src_ckpt["arch"])

        # ---- Load dataset cases + split for val ----
        cases = load_dataset_cases(dataset_ref)
        train_cases, val_cases = split_train_val(cases, val_frac=VAL_FRAC)
        print(
            f"  loaded {len(cases)} cases   "
            f"train={len(train_cases)}  val={len(val_cases)}"
        )

        # ---- Norm stats: derive (Stage 1) OR reuse from source ckpt ----
        if derive_stats_from_data:
            stats = derive_stats(train_cases)
            stats_source = "derived_from_synthetic"
        else:
            stats = dict(src_ckpt["stats"])
            for key, arr in stats.items():
                if isinstance(arr, np.ndarray):
                    stats[key] = arr.astype(np.float32)
            stats_source = f"inherited_from:{source_model_ref}"

        # ---- Build model, load state_dict ----
        model = TransolverPointFieldSurrogate(**arch).to(device)
        model.load_state_dict(src_ckpt["state_dict"])

        # ---- DataLoaders ----
        train_loader = DataLoader(
            HiLiftPointPatchDataset(
                train_cases, stats, PATCH_POINTS, PATCHES_PER_CASE,
                random_patches=True,
            ),
            batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        )
        val_loader = DataLoader(
            HiLiftPointPatchDataset(
                val_cases, stats, PATCH_POINTS,
                max(16, PATCHES_PER_CASE // 4),
                random_patches=False,
            ),
            batch_size=BATCH_SIZE, shuffle=False,
        )

        # ---- Optimiser + cosine schedule ----
        optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

        # ---- Training loop ----
        best_val_loss = float("inf")
        epoch_history: list[dict] = []
        global_step = 0
        for epoch in range(1, epochs + 1):
            model.train()
            batch_losses: list[float] = []
            for x, pos, cond, y in train_loader:
                x, pos, cond, y = x.to(device), pos.to(device), cond.to(device), y.to(device)
                optim.zero_grad(set_to_none=True)
                pred = model(x, pos, cond)
                loss = F.mse_loss(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optim.step()
                batch_losses.append(loss.item())
                global_step += 1
            scheduler.step()

            train_loss = float(np.mean(batch_losses))
            val_loss, val_mae, val_rmse = evaluate(model, val_loader, stats)
            best_val_loss = min(best_val_loss, val_loss)

            run.log({
                "epoch":                epoch,
                "train/loss":           train_loss,
                "val_metrics/loss":     val_loss,
                "val_metrics/mae":      val_mae,
                "val_metrics/rmse":     val_rmse,
                "learning_rate":        float(scheduler.get_last_lr()[0]),
            }, step=global_step)

            epoch_history.append({
                "epoch": epoch,
                "train/loss": train_loss,
                "val_metrics/loss": val_loss,
                "val_metrics/mae":  val_mae,
                "val_metrics/rmse": val_rmse,
            })

            print(
                f"  epoch {epoch:>2}: train={train_loss:.4f}   "
                f"val={val_loss:.4f}   mae={val_mae:.4g}   rmse={val_rmse:.4g}"
            )

        # ---- Save output checkpoint ----
        out_path = TMP_DIR / f"{stage}.pt"
        torch.save({
            "state_dict":     model.state_dict(),
            "arch":           arch,
            "stats":          stats,
            "stage":          stage,
            "source_model":   source_model_ref,
            "source_dataset": dataset_ref,
            "epoch_history":  epoch_history,
            "best_val_loss":  best_val_loss,
        }, out_path)

        # ---- Log + link to target Collection ----
        out_art = wandb.Artifact(
            name=f"hilift_transolverpp_{stage}",
            type="model",
            metadata={
                "stage":            stage,
                "arch":             arch,
                "stats_source":     stats_source,
                "source_model_ref": source_model_ref,
                "source_dataset":   dataset_ref,
                "epochs":           epochs,
                "best_val_loss":    best_val_loss,
                "created_at":       utc_ts(),
            },
        )
        out_art.add_file(str(out_path), name="checkpoint.pt")
        logged = run.log_artifact(out_art)
        logged.wait()

        target_path = f"wandb-registry-{MODELS_REG}/{target_collection}"
        run.link_artifact(logged, target_path=target_path, aliases=["production", "v1"])

        # ---- Write the Collection card (idempotent — safe to re-run) ----
        try:
            coll = wandb.Api().artifact_collection(
                type_name="model", name=target_path,
            )
            coll.description = collection_description
            coll.save()
            card_status = f"card written ({len(collection_description)} chars)"
        except Exception as exc:
            card_status = f"card skipped: {exc}"

        run.summary.update({
            "stage":                  stage,
            "val_metrics/best_loss":  best_val_loss,
            "linked_to":              f"{target_path}:production",
            "finished_at":            utc_ts(),
        })
        print(f"  linked -> {target_path}:production   ({card_status})")
        return f"{target_path}:production"


## 3. Stage 1 — init team

The first team's job. Consumes the untrained base model + the
`synthetic` dataset (LHC001 × 4 AoAs). This is the stage that **owns
the normalisation contract**: `derive_stats=True`, computing
`point_mean/std`, `target_mean/std`, and `aoa_mean/std` from the
training split of `synthetic` and baking them into the output
checkpoint's `stats` dict. Every downstream stage inherits these
stats unchanged — that's what makes chained fine-tuning safe under
Registry-only handoff.


In [ ]:
INITIALISED_CARD = """\
# HiLift multistage — Stage 1 output (`initialised`)

Model after Stage 1 (init team) of the multistage pipeline.
**Owns the normalisation contract** — `stats` (point / target / AoA
mean & std) derived from `synthetic` and baked into every
checkpoint here. Every downstream stage inherits these stats
unchanged.

**Upstream:** `wandb-registry-hilift_multistage_models/base` +
`wandb-registry-hilift_multistage_datasets/synthetic`.
**Downstream:** consumed by Stage 2 (pretraining team).

**Alias:** `production` moves forward with each re-run of Stage 1.
"""

print("=" * 66)
print("Stage 1 — init team")
print("=" * 66)

initialised_ref = train_one_stage(
    source_model_ref  = f"wandb-registry-{MODELS_REG}/base:production",
    dataset_ref       = f"wandb-registry-{DATASETS_REG}/synthetic:production",
    stage             = "initialised",
    target_collection = "initialised",
    epochs            = 3,
    lr                = 1e-3,
    derive_stats_from_data = True,
    collection_description = INITIALISED_CARD,
)
print(f"\nStage 1 output: {initialised_ref}")


## 4. Stage 2 — pretraining team

Consumes Stage 1's output (`initialised:production`) + the broader
`pretraining` dataset (all 3 geometries × [4°, 8°, 12°]).
**`derive_stats=False`** — Stage 2 reads Stage 1's stats from the
checkpoint and re-emits them unchanged in its own output. The
`stats_source` field in the output artifact's metadata records
this provenance: `inherited_from:<source ref>`.

This is the mechanism that lets multiple teams collaborate without
implicit contracts — the checkpoint IS the contract.


In [ ]:
PRETRAINED_CARD = """\
# HiLift multistage — Stage 2 output (`pretrained`)

Model after Stage 2 (pretraining team). Trained on the broader
9-case `pretraining` dataset (all 3 geometries × [4°, 8°, 12°]).
**Norm stats inherited unchanged** from `initialised` via the
checkpoint dict.

**Upstream:** `initialised:production` +
`hilift_multistage_datasets/pretraining:production`.
**Downstream:** consumed by Stage 3 (posttraining team).

Not a deployment target on its own — reads best as the middle
version of the pipeline.
"""

print("=" * 66)
print("Stage 2 — pretraining team")
print("=" * 66)

pretrained_ref = train_one_stage(
    source_model_ref  = initialised_ref,
    dataset_ref       = f"wandb-registry-{DATASETS_REG}/pretraining:production",
    stage             = "pretrained",
    target_collection = "pretrained",
    epochs            = 5,
    lr                = 5e-4,
    derive_stats_from_data = False,
    collection_description = PRETRAINED_CARD,
)
print(f"\nStage 2 output: {pretrained_ref}")


## 5. Stage 3 — posttraining team

The final fine-tune. Consumes Stage 2's output
(`pretrained:production`) + the `posttraining` dataset (all 3
geometries × [16°] — the high-AoA / near-stall regime that maps
onto Hodges' "aerodynamically interesting" domain). Same stats
propagation. Promotes to `posttrained:production`, which is the
canonical output of the pipeline: downstream consumers pull from
here.

Lower LR (`2e-4`) because we're fine-tuning a model that's already
seen the domain — no big steps needed.


In [ ]:
POSTTRAINED_CARD = """\
# HiLift multistage — Stage 3 output (`posttrained`, final)

Final pipeline output. Fine-tuned on the 3-case `posttraining`
dataset (all 3 geometries × [16°]) — the high-AoA / near-stall
regime. **This is the canonical deployment model** for the
multistage pipeline; pull from `:production` here.

**Upstream:** `pretrained:production` +
`hilift_multistage_datasets/posttraining:production`.
**Downstream:** deployment / evaluation. Norm stats inherited
from the pipeline root (`initialised`) via the checkpoint chain.

**Alias:** `production` — the stable public entry point for
downstream consumers.
"""

print("=" * 66)
print("Stage 3 — posttraining team")
print("=" * 66)

posttrained_ref = train_one_stage(
    source_model_ref  = pretrained_ref,
    dataset_ref       = f"wandb-registry-{DATASETS_REG}/posttraining:production",
    stage             = "posttrained",
    target_collection = "posttrained",
    epochs            = 4,
    lr                = 2e-4,
    derive_stats_from_data = False,
    collection_description = POSTTRAINED_CARD,
)
print(f"\nStage 3 output (final): {posttrained_ref}")


## 6. Lineage tracing — audit the whole pipeline

The W&B Registry keeps a lineage graph of every artifact edge —
which run produced each version, which runs consumed each version.
For a multistage pipeline that's the entire audit trail: given the
final `posttrained:production`, we can walk backwards through every
stage to the untrained base model and the datasets that fed it.

`print_lineage` below reconstructs that walk from Python: for each
model artifact we jump from Registry alias → `logged_by()` run →
inputs (`used_artifacts()`) → next model artifact, etc. The same
walk shows up interactively in the Registry Lineage tab of the W&B
UI.


In [ ]:
def print_lineage(final_ref: str) -> None:
    """Walk backwards from a Registry-aliased model, printing the chain."""
    api = wandb.Api()
    current_ref: str | None = final_ref
    depth = 0

    while current_ref is not None and depth < 10:  # safety fuse
        art = api.artifact(current_ref, type="model")
        producer = art.logged_by()
        producer_name = getattr(producer, "name", "(unknown)")
        producer_url = getattr(producer, "url", "")

        aliases = ", ".join(art.aliases) if art.aliases else "-"
        meta = art.metadata or {}
        stage_label = meta.get("stage", "?")

        print(f"[{depth}] {current_ref}")
        print(f"      stage       = {stage_label}")
        print(f"      aliases     = [{aliases}]")
        print(f"      produced by = {producer_name}   {producer_url}")

        # Walk to the next artifact in the chain via the source_model_ref
        # recorded in this artifact's metadata (set by train_one_stage).
        next_ref = meta.get("source_model_ref")
        dataset_ref = meta.get("source_dataset")
        if dataset_ref:
            print(f"      consumed    = {dataset_ref}")
        if next_ref:
            print(f"      source model:")
            current_ref = next_ref
        else:
            current_ref = None
        depth += 1
        print()


print_lineage(posttrained_ref)


## 7. How does this generalise — and what to look at in the W&B UI

The three-stage story above is deliberately small — three
sequential Registry hops on one architecture. The same pattern
scales along three axes:

**Multiple teams.** Each stage above is a `wandb.init` block that
could just as easily live in a separate Python file on a separate
machine. Team A owns Stage 1's job; team B owns Stage 2's; team C
owns Stage 3's. The Registry — and specifically the `production`
alias on each Collection — is the only coordination surface they
need. A team promotes its output; downstream teams re-pull and
continue. `production` moves; `v1`, `v2`, `v3` persist behind it
for rollback.

**Multiple architectures.** The `arch` dict on every checkpoint is
data-driven — `train_one_stage` reads it from the source model and
rebuilds the same shape. Run architecture sweeps as parallel
Collections: `base_hidden96`, `base_hidden128`, `base_hidden192`,
each with its own `initialised`, `pretrained`, `posttrained`
chain. Every arch family is a parallel lineage; the pipeline code
doesn't change, only the Registry paths.

**Multiple experiments within one family.** Ablations pile up as
`v2`, `v3` versions in the same Collection. Either move the
`production` alias when the winning ablation lands (downstream
picks it up automatically), or use named aliases like
`experiment-baseline`, `experiment-noaug` and have downstream teams
pin explicitly.

**What to look at in the W&B UI**

- **Runs table** — filter by `job_type=train-stage` to see the
  three stage runs in one view. Distinguish outputs by the `stage`
  summary key.
- **Registry** — under `hilift_multistage_models` you should now
  see four Collections (`base`, `initialised`, `pretrained`,
  `posttrained`), each with a version aliased `production` + `v1`.
- **Lineage graph** — click any Collection version and open the
  lineage view. `posttrained:production` traces back through
  `pretrained:v1` → `initialised:v1` → `base:v1`, plus dataset
  consumption edges into each. The `print_lineage` walk above
  reconstructs the same chain from Python.
